- Эта тетрадь посвящена выгрузке таблиц xlxs из бд
На данный момент есть:
1. Таблица с базовой структурой разметки (там все)
```
    'case_iD': [],
    
    'sent_ID_RU': [],
    'sent_text_RU': [],
    'concept_unit_RU': [],
    
    'token_ID': [],
    'token_RU': [],
    'token_pos_RU': [],
    'type_RU': [],
    'connotation_RU': [],
    'sentiment_RU': [],
    'gram_structure_RU': [],
    
    тоже самое для английского
    
    'translation_shift': [],
    'cosine_sim_LaBSE': [],
    'shift_notes': [],
    'verified': []
```

2. Страницы для грамматики
3. Страница для коннотаций, типов, метафор и прочего

In [1]:
import os

import pandas as pd
from openpyxl import Workbook, load_workbook

import sqlite3
from pathlib import Path

In [2]:
DB_PATH = Path("../db/olfactory.db")
conn = sqlite3.connect(DB_PATH)
FILENAME = '../results/Разметка.xlsx'

## Export xlsx и листов

In [22]:
def create_excel_with_merges(df, filename, title, overwrite=False):
    """Автоматически использует все колонки из df в их порядке"""
    # Проверяем, существует ли файл
    if os.path.exists(filename):
        wb = load_workbook(filename)
    else:
        wb = Workbook()                  # Создаем новый файл
        if 'Sheet' in wb.sheetnames:     # Удаляем дефолтный лист "Sheet"
            wb.remove(wb['Sheet'])
    
    if overwrite and title in wb.sheetnames:
        wb.remove(wb[title])
        print(f"🔄 Лист '{title}' перезаписан")
    else:
        # Проверяем, существует ли лист с таким названием
        original_title = title
        counter = 1
        while title in wb.sheetnames:
            title = f"{original_title}_{counter}"
            counter += 1
        
        if title != original_title:
            print(f"⚠️ Лист '{original_title}' уже существует, создан '{title}'")
        
    # Создаем новый лист (теперь точно с уникальным названием)
    ws = wb.create_sheet(title=title)
    
    # # Заголовки с объединениями
    # ws['A1'] = '🇷🇺 РУССКИЙ ОРИГИНАЛ'
    # ws['C1'] = '🇬🇧 АНГЛИЙСКИЙ ПЕРЕВОД'
    # ws.merge_cells('A1:B1')
    # ws.merge_cells('C1:D1')
    
    # Подзаголовки - просто берем имена колонок из df
    for col_idx, col_name in enumerate(df.columns, start=1):
        ws.cell(row=1, column=col_idx, value=col_name)
    
    # Данные
    for r, (_, row) in enumerate(df.iterrows(), start=2):
        for col_idx, value in enumerate(row, start=1):
            ws.cell(row=r, column=col_idx, value=value)
    
    wb.save(filename)
    print(f"✅ Создан лист '{title}' в {filename} с колонками: {list(df.columns)}")

## Создание df для разных страниц

### Подключаемся к бд для заполнения

In [23]:
# информация о таблицах
tables = pd.read_sql_query("SELECT name FROM sqlite_master WHERE type='table'", conn)
for table in ['texts', 'translations', 'sentences', 'alignment']:
    count = pd.read_sql_query(f"SELECT COUNT(*) as cnt FROM {table}", conn)
    # print(f"{table}: {count.iloc[0]['cnt']}")

# tables

### Лист с полной разметкой

In [24]:
# Создаем пустой DataFrame с нужной структурой колонок
df = pd.DataFrame({
    'case_iD': [],
    
    'sent_ID_RU': [],
    'sent_text_RU': [],
    'concept_unit_RU': [],
    
    'token_ID': [],
    'token_RU': [],
    'token_pos_RU': [],
    'type_RU': [],
    'connotation_RU': [],
    'sentiment_RU': [],
    'gram_structure_RU': [],
    
    'sent_ID_EN': [],
    'sent_text_EN': [],
    'concept_unit_EN': [],
    
    'token_ID_EN': [],
    'token_EN': [],
    'token_pos_EN': [],
    'type_EN': [],
    'connotation_EN': [],
    'sentiment_EN': [],
    'gram_structure_EN': [],
    
    'translation_shift': [],
    'cosine_sim_LaBSE': [],
    'shift_notes': [],
    'verified': []
})

### Лист для грамматики

In [25]:
import pandas as pd
import numpy as np

# Получаем данные из БД через alignment (только парные предложения)
df_gr = pd.read_sql_query("""
    SELECT  
        ru.sentence_id as sent_ID_RU,
        ru.sentence as sent_text_RU,
        en.sentence_id as sent_ID_EN,
        en.sentence as sent_text_EN
    FROM alignment a
    JOIN sentences ru ON ru.sentence_id = a.sentence_ru_id AND ru.language = 'ru'
    JOIN sentences en ON en.sentence_id = a.sentence_en_id AND en.language = 'en'
    ORDER BY ru.sentence_id
""", conn)

print(f"✅ Загружено {len(df_gr)} пар предложений через alignment")

# Теперь добавляем нужные колонки в нужном порядке
df_gr = df_gr[[
    'sent_ID_RU', 'sent_text_RU', 
    'sent_ID_EN', 'sent_text_EN'
]].copy()

# Добавляем пустые колонки в том порядке, который вы хотите
df_gr['concept_unit_RU'] = ''
df_gr['gram_structure_RU'] = ''
df_gr['concept_unit_EN'] = ''
df_gr['gram_structure_EN'] = ''
df_gr['translation_shift'] = ''
df_gr['cosine_sim_LaBSE'] = np.nan
df_gr['shift_notes'] = ''
df_gr['verified'] = ''

# Переставляем колонки в нужном порядке
column_order = [
    'sent_ID_RU', 'sent_text_RU', 'concept_unit_RU', 'gram_structure_RU',
    'sent_ID_EN', 'sent_text_EN', 'concept_unit_EN', 'gram_structure_EN',
    'translation_shift', 'cosine_sim_LaBSE', 'shift_notes', 'verified'
]

df_gr = df_gr[column_order]
# df_gr

✅ Загружено 162 пар предложений через alignment


### Лист для типа, коннотации и тональности

In [ ]:
# Объединяем
df_tok = pd.DataFrame({
    'sent_ID_RU': df_gr['sent_ID_RU'],
    'sent_text_RU': df_gr['sent_text_RU'],

    'token_RU':'',

    'sentiment_RU': '',
    'type_RU': '',  # есть ли оценочное слово?
    'is_metaphor_RU': '',     # метафора или буквально?
    
    
    'sent_ID_EN': df_gr['sent_ID_EN'],
    'sent_text_EN': df_gr['sent_text_EN'],
    
    'token_EN':'',

    'sentiment_EN': '',
    'type_EN': '',  # есть ли оценочное слово?
    'is_metaphor_EN': '',     # метафора или буквально?
})



## Создание листов

In [27]:
create_excel_with_merges(df, FILENAME, 'Анализ')
create_excel_with_merges(df_gr, FILENAME, 'Разметка')
create_excel_with_merges(df_tok, FILENAME, 'Качество')

✅ Создан лист 'Анализ' в ../results/Разметка.xlsx с колонками: ['case_iD', 'sent_ID_RU', 'sent_text_RU', 'concept_unit_RU', 'token_ID', 'token_RU', 'token_pos_RU', 'type_RU', 'connotation_RU', 'sentiment_RU', 'gram_structure_RU', 'sent_ID_EN', 'sent_text_EN', 'concept_unit_EN', 'token_ID_EN', 'token_EN', 'token_pos_EN', 'type_EN', 'connotation_EN', 'sentiment_EN', 'gram_structure_EN', 'translation_shift', 'cosine_sim_LaBSE', 'shift_notes', 'verified']
✅ Создан лист 'Разметка' в ../results/Разметка.xlsx с колонками: ['sent_ID_RU', 'sent_text_RU', 'concept_unit_RU', 'gram_structure_RU', 'sent_ID_EN', 'sent_text_EN', 'concept_unit_EN', 'gram_structure_EN', 'translation_shift', 'cosine_sim_LaBSE', 'shift_notes', 'verified']
✅ Создан лист 'Качество' в ../results/Разметка.xlsx с колонками: ['sent_ID_RU', 'sent_text_RU', 'token_RU', 'sentiment_RU', 'has_evaluation_RU', 'is_metaphor_RU', 'sent_ID_EN', 'sent_text_EN', 'token_EN', 'sentiment_EN', 'has_evaluation_EN', 'is_metaphor_EN']


### Чтение, агрегации и прочие тесты

In [28]:
# 1. Читаем существующий Excel
df = pd.read_excel(FILENAME, sheet_name='Анализ', header=0)
# обратить внимание на header. Если строк с заголовками будет больше, то нучно ставить 1. Потому что заголовки на 2 строчке, а не 1. 
df 

,case_iD,sent_ID_RU,sent_text_RU,concept_unit_RU,token_ID,token_RU,token_pos_RU,type_RU,connotation_RU,sentiment_RU,...,token_EN,token_pos_EN,type_EN,connotation_EN,sentiment_EN,gram_structure_EN,translation_shift,cosine_sim_LaBSE,shift_notes,verified


In [4]:
conn = sqlite3.connect(DB_PATH)

ru = pd.read_sql(f"""
    SELECT sentence_id, sentence, concept_phrase, gram_structure
    FROM sentences 
    WHERE source_type='original' AND text_id=1 AND language='ru'
    ORDER BY position
""", conn)

In [15]:
def get_data(orig_text_id, translation_id):
    """Загружает данные из БД"""
    conn = sqlite3.connect(DB_PATH)
    
    orig = pd.read_sql(f"""
        SELECT sentence_id AS ru_id, position, sentence,
            search_word AS token_text, concept_phrase, gram_structure
        FROM sentences
        WHERE source_type='original' AND text_id={orig_text_id} AND language='ru'
        ORDER BY position
    """, conn)
    
    translation = pd.read_sql(f"""
        SELECT sentence_id AS en_id, position, sentence,
            search_word AS token_text, concept_phrase, gram_structure
        FROM sentences
        WHERE source_type='translation' AND translation_id={translation_id} AND language='en'
        ORDER BY position
    """, conn)
    
    align = pd.read_sql(f"""
        SELECT a.sentence_ru_id AS ru_id,
            a.sentence_en_id AS en_id,
            a.cosine_sim,
            COALESCE(a.auto_aligned, 0) AS auto_aligned
        FROM alignment a
        JOIN sentences sr ON sr.sentence_id = a.sentence_ru_id
        JOIN sentences se ON se.sentence_id = a.sentence_en_id
        WHERE sr.text_id = {orig_text_id} AND se.translation_id = {translation_id}
    """, conn)
    
    conn.close()
    
    # Нумеруем
    orig.insert(0, '№', range(1, len(orig) + 1))
    translation.insert(0, '№', range(1, len(translation) + 1))

    # ru_id_to_num = dict(zip(ru['sentence_id'], ru['ru_num']))
    # ru_num_to_text = dict(zip(ru['ru_num'], ru['sentence']))
    # ru_num_to_concept = dict(zip(ru['ru_num'], ru['concept_phrase']))
    # ru_num_to_grammar = dict(zip(ru['ru_num'], ru['gram_structure']))
    
    return orig, translation, align


In [23]:
orig, translation, align = get_data(1, 1)

In [24]:
df_all = (
    translation
    .merge(align, on='en_id', how='outer')                              # сохраняем все EN + все связи
    .merge(orig[['ru_id', 'sentence']], on='ru_id', how='outer', suffixes=('_en', '_ru'))  # сохраняем все RU
)

In [ ]:
# Выбираем колонки в нужном порядке
en_cols = [
    'en_id', 
    'sentence_en', 
    'token_text', 
    'ru_id',       
    'sentence_ru', 
    'cosine_sim', 
    'auto_aligned'
]

# Применяем
df_all = df_all[en_cols]

In [27]:
df_all

,en_id,position,sentence_en,token_text,ru_id,sentence_ru,cosine_sim,auto_aligned
0,212.0,623.0,The stuffy rooms smelled of mint .,smelled,1.0,В душных комнатах пахло мятой .,0.791446,1.0
1,213.0,724.0,In the heat their red flowers and prickly leav...,smell,2.0,Красные его цветы и листья с колючками издавал...,0.836972,1.0
2,215.0,761.0,"Misty stoneware jugs of ice cold milk , wet ma...",smell,3.0,Запотевшие кувшины — глечики — с ледяным молок...,0.833100,1.0
3,217.0,856.0,The bar gave off the faintest smell of roses .,smell,4.0,Брусок издавал тончайший запах роз .,0.804614,1.0
4,219.0,874.0,The cottage smelled of warm milk .,smelled,5.0,В хате пахло топленым молоком .,0.833562,1.0
...,...,...,...,...,...,...,...,...
278,399.0,15721.0,Soon the paddles once more lazily dipped into ...,smell,NaN,NaN,NaN,NaN
279,409.0,16507.0,"Thus , he was always having to cover his track...",scent,NaN,NaN,NaN,NaN
280,430.0,18255.0,"We got out and followed Pan Sotnik , stragglin...",stinking,NaN,NaN,NaN,NaN
281,431.0,18601.0,It had the sour smell of paint and the particu...,smell,NaN,NaN,NaN,NaN


In [ ]:

# Используем merge:
df_en = df_en.merge(
    en_info[['en_id', 'ru_id', 'ru_text', 'sim', 'авто', 'translation_shift']],
    on='en_id',
    how='left'  # сохраняем все английские, даже без пары
)


# В map добавляем ru_id и ru_text (вместо ru_№)
en_info_map = en_info.set_index('en_id')[['ru_id', 'ru_text', 'sim', 'авто', 'translation_shift']]
df_en = df_en.join(en_info_map, on='en_id')

# Заполняем пустые значения (теперь для ru_id и ru_text)
df_en['ru_id'] = df_en['ru_id'].fillna('')
df_en['ru_text'] = df_en['ru_text'].fillna('')
df_en['translation_shift'] = df_en['translation_shift'].fillna('')

# ── Статистика ────────────────────────────────────────────────────────────────
aligned_count = align.shape[0]

# Считаем русские без пары (по ru_id из align)
ru_with_pairs = set(align['ru_id'])
unaligned_ru = len(df_ru[~df_ru['ru_id'].isin(ru_with_pairs)])

# Считаем английские без пары (по en_id из align)
en_with_pairs = set(align['en_id'])
unaligned_en = len(df_en[~df_en['en_id'].isin(en_with_pairs)])

print(f"   RU предложений:  {len(df_ru)} (без пары: {unaligned_ru})")
print(f"   EN предложений:  {len(df_en)} (без пары: {unaligned_en})")
print(f"   Уже выровнено:   {aligned_count} пар")

# ── 7. Запись в Excel ──────────────────────────────────────────────────────
Path(output_path).parent.mkdir(parents=True, exist_ok=True)

ru_cols = ['ru_id', 'sentence', 'token_text']
en_cols = ['en_id', 'sentence', 'token_text', 'ru_id', 'ru_text', 'sim', 'авто', 'translation_shift']

with pd.ExcelWriter(output_path, engine='openpyxl') as writer:
    df_en[en_cols].to_excel(writer, sheet_name='Английские', index=False)
    df_ru[ru_cols].to_excel(writer, sheet_name='Русские', index=False)